# Learning 3: Simple Chains

**Goal**: Connect components together using the LCEL pipe operator

## What You'll Learn
- What chains are and why they're useful
- Using the `|` pipe operator
- Chaining prompts, LLMs, and output parsers
- Creating multi-step chains

In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini")
print("Setup complete!")

/Users/syedraza/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Setup complete!


## What is a Chain?

A chain connects multiple components:

```
[Input] → [Prompt] → [LLM] → [Output Parser] → [Result]
```

Instead of calling each component separately, chains let you combine them into a single callable.

## The Pipe Operator `|`

LangChain uses the pipe operator `|` to chain components. It's called LCEL (LangChain Expression Language).

In [6]:
# Without chaining (verbose) - Manual step-by-step
prompt = ChatPromptTemplate.from_template("Translate '{text}' to {language}")
formatted = prompt.format_messages(text="Hello, how are you?", language="Spanish")
response = llm.invoke(formatted)
text = response.content

print("Without chain (manual steps):", text)

Without chain (manual steps): 'Hello, how are you?' in Spanish is 'Hola, ¿cómo estás?'


In [21]:
# With chaining (clean!) - All in one line
prompt = ChatPromptTemplate.from_template("Translate '{text}' to {language}")
output_parser = StrOutputParser()

chain = prompt | llm | output_parser

result = chain.invoke({"text": "Hello, how are you?", "language": "French"})
print("With chain (one call):", result)

With chain (one call): 'Hello, how are you?' translates to 'Bonjour, comment ça va?' in French.


## Understanding StrOutputParser

The `StrOutputParser` extracts just the text content from the LLM response.

In [22]:
# Without parser - returns AIMessage object
fact_prompt = ChatPromptTemplate.from_template("Tell me a fact about {topic}")
chain_no_parser = fact_prompt | llm
result = chain_no_parser.invoke({"topic": "birds"})
print("Type:", type(result))
print("Result:", result)

Type: <class 'langchain_core.messages.ai.AIMessage'>
Result: content='Birds are the only animals with feathers, which are essential for insulation, waterproofing, and aiding in flight. There are approximately 10,000 known species of birds, showcasing a wide range of colors, sizes, and behaviors.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 13, 'total_tokens': 60, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c4585b5b9c', 'id': 'chatcmpl-D1wEPTGXwiW9WYM9INbKTUxF6op5Q', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--e0a7527c-8cfa-468b-ae6b-8b1fc89885af-0' usage_metadata={'input_tokens': 13, 'output_tokens': 47, 'total_tokens': 60, 'input_token_details': {'audio': 0, 

In [23]:
# With parser - returns just the string
chain_with_parser = fact_prompt | llm | StrOutputParser()
result = chain_with_parser.invoke({"topic": "birds"})
print("Type:", type(result))
print("Result:", result)

Type: <class 'str'>
Result: Birds are the only animals with feathers, which are unique to their class, Aves. Feathers serve various functions, including insulation, waterproofing, and most notably, aiding in flight. They come in different types and serve multiple purposes, from providing camouflage to attracting mates.


## Multi-Step Chain

Chain the output of one LLM call as input to another.

In [24]:
# Step 1: Extract key points from text
extract_prompt = ChatPromptTemplate.from_template(
    "Extract 3 key points from this text in a bullet list:\n\n{text}"
)

# Step 2: Summarize those points
summarize_prompt = ChatPromptTemplate.from_template(
    "Write a one-sentence summary based on these points:\n\n{key_points}"
)

# Chain step 1
extract_chain = extract_prompt | llm | StrOutputParser()

# Test step 1
sample_text = "Artificial intelligence is transforming industries. Machine learning enables computers to learn from data. Deep learning uses neural networks to solve complex problems."
key_points = extract_chain.invoke({"text": sample_text})
print("Key points extracted:")
print(key_points)

Key points extracted:
- Artificial intelligence is transforming various industries.
- Machine learning allows computers to learn from data.
- Deep learning utilizes neural networks to address complex problems.


In [25]:
# Combine both chains - extract then summarize
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Full pipeline: text → key points → summary
full_chain = (
    {"key_points": extract_prompt | llm | StrOutputParser()}
    | summarize_prompt
    | llm
    | StrOutputParser()
)

new_text = "Cloud computing provides on-demand access to computing resources. It offers scalability and cost efficiency. Major providers include AWS, Azure, and Google Cloud."
summary = full_chain.invoke({"text": new_text})
print("Summary:", summary)

Summary: Cloud computing delivers on-demand access to scalable and cost-efficient computing resources, with major providers including AWS, Azure, and Google Cloud.


## Parallel Chains with RunnableParallel

Run multiple chains at once and combine their outputs.

In [14]:
from langchain_core.runnables import RunnableParallel

# Create parallel chains
joke_prompt = ChatPromptTemplate.from_template("Tell a short joke about {topic}")
fact_prompt = ChatPromptTemplate.from_template("Tell a fact about {topic}")

parallel_chain = RunnableParallel(
    joke=joke_prompt | llm | StrOutputParser(),
    fact=fact_prompt | llm | StrOutputParser()
)

result = parallel_chain.invoke({"topic": "coffee"})
print("Joke:", result["joke"])
print("\nFact:", result["fact"])

Joke: Why do coffee beans never get in trouble?  

Because they know how to espresso themselves!

Fact: One interesting fact about coffee is that it is actually a fruit! The coffee beans we use to make the beverage are the seeds of the coffee cherry, which is a small, red or purple fruit that grows on coffee plants. Each cherry typically contains two seeds (the coffee beans) inside.


## Adding Custom Functions with RunnableLambda

In [15]:
from langchain_core.runnables import RunnableLambda

# Custom function to process output
def make_uppercase(text: str) -> str:
    return text.upper()

def add_emoji(text: str) -> str:
    return f"✨ {text} ✨"

# Chain with custom functions
chain = (
    ChatPromptTemplate.from_template("Say hello to {name}")
    | llm
    | StrOutputParser()
    | RunnableLambda(make_uppercase)
    | RunnableLambda(add_emoji)
)

result = chain.invoke({"name": "Alice"})
print(result)

✨ HELLO, ALICE! HOW ARE YOU TODAY? ✨


## Streaming with Chains

Chains support streaming too!

In [17]:
chain = (
    ChatPromptTemplate.from_template("Write a short product description for {product}")
    | llm
    | StrOutputParser()
)

print("Streaming product description:")
for chunk in chain.stream({"product": "a smart water bottle that tracks hydration"}):
    print(chunk, end="", flush=True)
print("\n")

Streaming product description:
Introducing theIntroducing the Hydrate HydrateSmart WaterSmart Water Bottle – Bottle – your ultimate your ultimate hydration companion hydration companion! This! This innovative smart innovative smart water bottle water bottle features advanced features advanced technology to technology to track your track your fluid intake fluid intake, reminding, reminding you to you to stay hydrated stay hydrated throughout the throughout the day. day. With a With a sleek, sleek, ergonomic design ergonomic design and an and an easy-to easy-to-read LED-read LED display, display, it not it not only keeps only keeps your beverages your beverages at the at the perfect temperature perfect temperature but also but also syncs syncs seamlessly with seamlessly with your smartphone your smartphone via a via a dedicated app dedicated app. Set. Set personalized hydration personalized hydration goals, goals, receive reminders receive reminders, and, and monitor your monitor your pr

## Real-World Example: Content Moderator + Rewriter

Let's build a practical chain that:
1. Analyzes text for tone/issues
2. Decides if it needs rewriting
3. Rewrites if needed

In [18]:
from pydantic import BaseModel, Field

# Step 1: Analyze the text
class TextAnalysis(BaseModel):
    is_appropriate: bool = Field(description="Whether the text is professional and appropriate")
    reason: str = Field(description="Brief reason for the assessment")

analyze_prompt = ChatPromptTemplate.from_template(
    "Analyze if this text is professional and appropriate for a business email:\n\n{text}\n\n"
    "Consider tone, language, and professionalism."
)

analyze_chain = analyze_prompt | llm.with_structured_output(TextAnalysis)

# Test the analyzer
sample_text = "Hey dude, your idea is kinda stupid tbh. We should do something else."
analysis = analyze_chain.invoke({"text": sample_text})
print(f"Appropriate: {analysis.is_appropriate}")
print(f"Reason: {analysis.reason}")

Appropriate: False
Reason: The tone is overly casual and informal, using slang ('dude', 'kinda stupid', 'tbh') which is not suitable for a professional business email. It also lacks constructive feedback and is dismissive.


In [19]:
# Step 2: Rewrite inappropriate text
rewrite_prompt = ChatPromptTemplate.from_template(
    "Rewrite this text to be professional and appropriate for a business email:\n\n{text}"
)

rewrite_chain = rewrite_prompt | llm | StrOutputParser()

# Test the rewriter
rewritten = rewrite_chain.invoke({"text": sample_text})
print("Original:", sample_text)
print("\nRewritten:", rewritten)

Original: Hey dude, your idea is kinda stupid tbh. We should do something else.

Rewritten: Subject: Suggestions for Improvement

Hi [Recipient's Name],

I hope this message finds you well. I wanted to provide some feedback on your recent idea. While I appreciate the effort you put into it, I believe we might want to explore alternative options that could be more effective.

Let’s discuss this further and brainstorm some different approaches together.

Best regards,

[Your Name]  
[Your Position]  
[Your Company]  
[Your Contact Information]  


In [20]:
# Step 3: Combine both into a smart chain with conditional logic
def smart_moderate(input_dict):
    """Analyzes text and only rewrites if inappropriate"""
    text = input_dict["text"]
    
    # Analyze
    analysis = analyze_chain.invoke({"text": text})
    
    # If appropriate, return original; otherwise rewrite
    if analysis.is_appropriate:
        return {
            "original": text,
            "result": text,
            "was_rewritten": False,
            "reason": analysis.reason
        }
    else:
        rewritten = rewrite_chain.invoke({"text": text})
        return {
            "original": text,
            "result": rewritten,
            "was_rewritten": True,
            "reason": analysis.reason
        }

# Wrap in RunnableLambda to make it chainable
moderator_chain = RunnableLambda(smart_moderate)

# Test with inappropriate text
print("Test 1 - Inappropriate:")
result1 = moderator_chain.invoke({"text": "Hey dude, your idea is kinda stupid tbh."})
print(f"Rewritten: {result1['was_rewritten']}")
print(f"Reason: {result1['reason']}")
print(f"Result: {result1['result']}")

print("\n" + "="*60 + "\n")

# Test with appropriate text
print("Test 2 - Appropriate:")
result2 = moderator_chain.invoke({"text": "Thank you for your proposal. I have some suggestions for improvement."})
print(f"Rewritten: {result2['was_rewritten']}")
print(f"Reason: {result2['reason']}")
print(f"Result: {result2['result']}")

Test 1 - Inappropriate:
Rewritten: True
Reason: The use of informal language ('Hey dude', 'kinda stupid', 'tbh') is unprofessional and could be perceived as disrespectful in a business context. A more respectful and constructive tone is needed.
Result: Subject: Feedback on Your Idea

Dear [Recipient's Name],

I hope this message finds you well. I wanted to share my thoughts on the idea you proposed. While I appreciate your creativity and initiative, I believe there may be some aspects that could benefit from further consideration.

I would be happy to discuss this in more detail and explore alternative approaches together. 

Thank you for your understanding.

Best regards,  
[Your Name]  
[Your Job Title]  
[Your Company]  


Test 2 - Appropriate:
Rewritten: True
Reason: The use of informal language ('Hey dude', 'kinda stupid', 'tbh') is unprofessional and could be perceived as disrespectful in a business context. A more respectful and constructive tone is needed.
Result: Subject: Feed

## Exercise: Build Your Own Chain

1. Create a chain that translates text and then summarizes it
2. Create parallel chains for different types of analysis
3. Add a custom function to your chain

In [ ]:
# Exercise Solution Example: Email Marketing Chain
# This chain generates personalized email subject lines and body content

from langchain_core.runnables import RunnableParallel

# Step 1: Create prompts for subject line and email body
subject_prompt = ChatPromptTemplate.from_template(
    "Write a compelling email subject line for a {product_type} aimed at {audience}. "
    "Make it attention-grabbing but professional."
)

body_prompt = ChatPromptTemplate.from_template(
    "Write a 3-sentence marketing email for a {product_type} targeting {audience}. "
    "Highlight key benefits and include a call-to-action."
)

# Step 2: Create parallel chains for subject and body
email_chain = RunnableParallel(
    subject=subject_prompt | llm | StrOutputParser(),
    body=body_prompt | llm | StrOutputParser()
)

# Step 3: Add custom formatting function
def format_email(result: dict) -> str:
    """Format the email with proper structure"""
    return f"""
📧 EMAIL PREVIEW
{'='*50}
Subject: {result['subject']}
{'='*50}

{result['body']}

{'='*50}
"""

# Step 4: Complete chain with formatting
complete_email_chain = email_chain | RunnableLambda(format_email)

# Test the chain
print("Generating marketing email...\n")
email = complete_email_chain.invoke({
    "product_type": "AI-powered fitness app",
    "audience": "busy professionals"
})
print(email)

# Try another example
print("\n" + "🔄 Generating another email...\n")
email2 = complete_email_chain.invoke({
    "product_type": "eco-friendly water bottle",
    "audience": "environmentally conscious millennials"
})
print(email2)

## Key Takeaways

1. Use `|` to chain components together
2. `StrOutputParser()` extracts text from LLM responses
3. `RunnableParallel` runs multiple chains simultaneously
4. `RunnableLambda` wraps custom functions for use in chains
5. Chains support `.invoke()`, `.stream()`, and `.batch()`

**Next**: Learning 4 - Tools Basics

## Bonus: Advanced Multi-Chain Example

A real-world scenario combining multiple chain patterns for a **Product Review Analyzer**.

This example demonstrates:
- Sequential chains (analyze → score → decide)
- Parallel chains (multiple analysis types)
- Conditional logic (different actions based on scores)
- Custom formatting

In [ ]:
# Multi-Chain Product Review Analyzer
from pydantic import BaseModel, Field
from langchain_core.runnables import RunnableParallel, RunnableLambda

# Step 1: Define structured output for analysis
class ReviewAnalysis(BaseModel):
    sentiment: str = Field(description="positive, negative, or neutral")
    score: int = Field(description="Score from 1-10")
    key_issues: list[str] = Field(description="List of main concerns or praises")

# Step 2: Create parallel chains for different analysis aspects
sentiment_prompt = ChatPromptTemplate.from_template(
    "Analyze the sentiment and score (1-10) of this product review. "
    "Also list key issues or praises:\n\n{review}"
)

summary_prompt = ChatPromptTemplate.from_template(
    "Create a one-line summary of this review:\n\n{review}"
)

category_prompt = ChatPromptTemplate.from_template(
    "What product category does this review belong to? "
    "Answer in 1-3 words:\n\n{review}"
)

# Parallel analysis chains
analysis_chain = RunnableParallel(
    sentiment_analysis=sentiment_prompt | llm.with_structured_output(ReviewAnalysis),
    summary=summary_prompt | llm | StrOutputParser(),
    category=category_prompt | llm | StrOutputParser()
)

# Step 3: Sequential decision chain based on score
def generate_response(analysis_result):
    """Generate appropriate response based on analysis"""
    sentiment_data = analysis_result["sentiment_analysis"]
    summary = analysis_result["summary"]
    category = analysis_result["category"]
    
    # Decision logic based on score
    if sentiment_data.score >= 8:
        action = "✅ PROMOTE: Feature in testimonials"
        response_template = "Thank you for your amazing feedback!"
    elif sentiment_data.score >= 5:
        action = "📊 MONITOR: Standard response"
        response_template = "Thank you for your review. We appreciate your feedback."
    else:
        action = "🚨 URGENT: Customer service follow-up needed"
        response_template = "We're sorry to hear about your experience. Our team will contact you shortly."
    
    # Generate personalized response
    response_prompt = ChatPromptTemplate.from_template(
        f"{response_template}\n\n"
        "Address these specific points from their review: {key_issues}\n"
        "Keep it brief and professional (2-3 sentences)."
    )
    
    response_chain = response_prompt | llm | StrOutputParser()
    response = response_chain.invoke({"key_issues": ", ".join(sentiment_data.key_issues)})
    
    return {
        "category": category,
        "sentiment": sentiment_data.sentiment,
        "score": sentiment_data.score,
        "key_issues": sentiment_data.key_issues,
        "summary": summary,
        "action": action,
        "response": response
    }

# Step 4: Complete multi-chain pipeline
review_pipeline = analysis_chain | RunnableLambda(generate_response)

# Step 5: Format final output
def format_report(result):
    return f"""
{'='*60}
📦 PRODUCT REVIEW ANALYSIS REPORT
{'='*60}

Category: {result['category']}
Sentiment: {result['sentiment'].upper()}
Score: {result['score']}/10

Summary: {result['summary']}

Key Points:
{chr(10).join(f"  • {issue}" for issue in result['key_issues'])}

{result['action']}

Suggested Response:
{'-'*60}
{result['response']}
{'='*60}
"""

final_chain = review_pipeline | RunnableLambda(format_report)

print("Analyzing product reviews...\n")

In [ ]:
# Test with positive review
positive_review = """
I absolutely love this wireless keyboard! The battery life is incredible - 
I've been using it for 3 months without charging. The typing feel is smooth 
and quiet, perfect for my home office. Setup was easy and the Bluetooth 
connection is stable. Highly recommend!
"""

result1 = final_chain.invoke({"review": positive_review})
print(result1)

In [ ]:
# Test with negative review
negative_review = """
Very disappointed with this laptop. It overheats constantly, even with light use. 
The battery drains in 2 hours despite claims of 8-hour battery life. Customer 
support was unhelpful when I reached out. The keyboard feels cheap and several 
keys are already sticking after just 2 weeks. Not worth the price at all.
"""

result2 = final_chain.invoke({"review": negative_review})
print(result2)

In [ ]:
# Test with mixed review
mixed_review = """
The phone camera is excellent - takes stunning photos in daylight. However, 
the battery life is mediocre and the phone gets warm during gaming. The screen 
is beautiful but I wish it came with a case. Overall decent for the price point.
"""

result3 = final_chain.invoke({"review": mixed_review})
print(result3)